# Project 1 | Sentinel-2 Land-Cover Classification (EuroSAT)

**EUMETSAT Junior ML Engineer — practice project**

**Goal:** classify Sentinel-2 RGB image patches into one of 10 land-cover classes using PyTorch. Build a *reproducible proof of concept*: controlled split, model training, held-out evaluation, and saved weights.

**Important:** This is **land-cover classification, not cloud detection**. The EuroSAT RGB version has 27,000 labeled 64×64 pixel patches. The standard random split is a learning benchmark and does not prove geographic generalization. For stronger future work, use a spatially independent split and/or multispectral bands.

Dataset: [EuroSAT original publication/data](https://zenodo.org/records/7711810). TorchVision loader is used here.

**Start:** Google Colab → Runtime → Change runtime type → GPU (if available). Execute cells in order. Results are produced only when you execute them.

In [ ]:
# Install if needed (Colab generally already has most packages)
%pip -q install torch torchvision scikit-learn matplotlib pandas pillow

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import EuroSAT
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)

## 1. Load the public dataset

The model receives **RGB** only (not all 13 Sentinel-2 bands). Different transforms are used for training and evaluation. The download happens when you run this notebook.

In [ ]:
ROOT = './data'
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
])
eval_tf = transforms.ToTensor()

full_eval = EuroSAT(root=ROOT, download=True, transform=eval_tf)
full_train = EuroSAT(root=ROOT, download=False, transform=train_tf)
labels = np.asarray(full_eval.targets)
classes = full_eval.classes
print('Images:', len(full_eval), 'Classes:', len(classes))
print(classes)

In [ ]:
indices = np.arange(len(full_eval))
train_idx, remaining = train_test_split(indices, test_size=0.30, random_state=SEED, stratify=labels)
val_idx, test_idx = train_test_split(remaining, test_size=0.50, random_state=SEED,
                                     stratify=labels[remaining])
train_ds = Subset(full_train, train_idx.tolist())
val_ds = Subset(full_eval, val_idx.tolist())
test_ds = Subset(full_eval, test_idx.tolist())
print('Train / validation / test:', len(train_ds), len(val_ds), len(test_ds))

BATCH_SIZE = 128
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                      pin_memory=torch.cuda.is_available())
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for ax, ix in zip(axes.flat, train_idx[:10]):
    img, lab = full_eval[int(ix)]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[lab], fontsize=9)
    ax.axis('off')
plt.suptitle('EuroSAT Sentinel-2 RGB samples')
plt.tight_layout()
plt.show()

## 2. Model and training

We use a modest convolutional neural network (CNN). It is purposefully small enough to understand. Do not claim this is a state-of-the-art solution. Validation data select the best epoch; the test set is used once for final assessment.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128 * 8 * 8, 128),
                                  nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.head(self.features(x))

model = SmallCNN(num_classes=len(classes)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def run_epoch(loader, train=False):
    model.train(mode=train)
    running_loss, correct, count = 0.0, 0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, targets)
            if train:
                loss.backward()
                optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        count += images.size(0)
    return running_loss/count, correct/count

EPOCHS = 6   # If validation keeps improving, try 10-12
history = []
best_val = -1
best_path = Path('eurosat_best_cnn.pt')
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_dl, train=True)
    val_loss, val_acc = run_epoch(val_dl, train=False)
    history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc))
    print(f'Epoch {epoch:02d}: train accuracy={train_acc:.3f}, validation accuracy={val_acc:.3f}')
    if val_acc > best_val:
        best_val = val_acc
        torch.save({'model_state': model.state_dict(), 'classes': classes,
                    'seed': SEED, 'validation_accuracy': val_acc}, best_path)
print('Best checkpoint:', best_path)

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist.epoch, hist.train_loss, label='train')
axes[0].plot(hist.epoch, hist.val_loss, label='validation')
axes[0].set_title('Cross-entropy loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist.epoch, hist.train_acc, label='train')
axes[1].plot(hist.epoch, hist.val_acc, label='validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig('eurosat_training_curves.png', dpi=150); plt.show()

## 3. Held-out test set

The model checkpoint is selected **without looking at the test result**. Report class-level metrics and mistakes, not only overall accuracy.

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_dl:
        pred = model(x.to(DEVICE)).argmax(dim=1).cpu().numpy()
        y_true.extend(y.numpy().tolist())
        y_pred.extend(pred.tolist())
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=classes,
                                        xticks_rotation=65, cmap='Blues', ax=ax,
                                        colorbar=False)
plt.tight_layout(); plt.savefig('eurosat_confusion_matrix.png', dpi=150); plt.show()
print('Test accuracy:', np.mean(np.array(y_true) == np.array(y_pred)))

## 4. Reproducibility & interview notes

When you finish: save notebook, trained weights, figures, a concise English README, and a requirements list in one GitHub repository. **Never upload the 27,000 source images** to your own repository; cite the dataset and provide the download instructions.

Be ready to explain: (1) labels vs pixels, (2) train/validation/test, (3) convolutional filters, (4) cross-entropy, (5) overfitting, (6) confusion matrix, (7) why a geographic hold-out is more rigorous than a random split, (8) what would change for 13-band data.

## 5. Error analysis
Investigate the most common semantic confusion in this run: Highway → River. Interpret this only as a hypothesis, not a causal diagnosis.


In [ ]:
# Error analysis: highway patches misclassified as rivers
import numpy as np
import matplotlib.pyplot as plt

true_labels = np.asarray(y_true)
pred_labels = np.asarray(y_pred)
highway = classes.index('Highway')
river = classes.index('River')
errors = np.where((true_labels == highway) & (pred_labels == river))[0]
print('Highway classified as River:', len(errors))

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, ix in zip(axes.flat, errors[:6]):
    image, _ = test_ds[int(ix)]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title('Actual: Highway | Predicted: River', fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(errors[:6]):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig('eurosat_error_analysis.png', dpi=180)
plt.show()
